# 04 — Golden RAGAS Dataset (50-row pilot · `gpt-4o` constructor)

> **A/B variant.** This notebook uses **`gpt-4o`** via the OpenAI API as the constructor.
> Companion notebook [`04_golden_dataset_gptoss.ipynb`](04_golden_dataset_gptoss.ipynb) runs the **same 50 questions** with `openai/gpt-oss-120b` via Groq. Comparing both lets us pick the constructor based on measured accept rate × cost rather than theoretical capability.

## Phase 3 — Stage 0 (mandatory smoke pilot)

**Inputs:**
- `data/processed/medqa_4opt.parquet` — 12,723 rows; we sample 50 stratified (same seed = 42, identical sample to gptoss variant)
- `data/processed/chunks.parquet` — chunk text lookup
- `data/indices/chroma_textbooks/` + `data/indices/bm25.pkl` — Hybrid retrieval (built in Notebook 02)

**Outputs (variant-specific):**
- `data/processed/golden_ragas_50_pilot_gpt4o.jsonl` — accepted rows
- `data/processed/golden_ragas_50_pilot_gpt4o_needs_review.jsonl`
- `data/processed/golden_ragas_50_pilot_gpt4o_dropped.jsonl`

**Pipeline (per question):**

```
Question (MedQA 4-opt sample, same seed as gptoss variant)
  └─► Hybrid RRF (Chroma + BM25, k=60)  → 10 candidate chunks
        └─► Pass 1 (gpt-4o, T=0)         → select 1-3 + keywords + sufficient?
              └─► Pass 2 (T=0.2)          → reference_answer + explanation + check_points + question_type + requires_multihop
                    └─► Pass 3 (T=0)      → 0–5 scores + final_status
                          └─► Stage F audit (pure Python)
                                └─► Stage G save (jsonl × 3)
```

**Decisions applied** (from [plan.md §0](../plan.md), original locked plan before the 2026-05-04 swap):

| # | Setting | Value |
|---|---|---|
| 6 | Hybrid fusion | RRF k=60 |
| 10 | Constructor (this variant) | `gpt-4o` via OpenAI API |
| — | JSON mode | Instructed JSON (same prompts as gptoss variant — kept identical for clean A/B) |

**Quality gates ([docs/todo.md §4](../docs/todo.md)):**
- Accept rate ≥ 80 %
- JSON malformation < 5 %
- Pass-1 sufficiency rate ≥ 90 %
- All cited `chunk_id`s resolvable in `chunks.parquet`
- `requires_multihop = "yes"` rate < 60 %

**Cost expectation (measured in §10):**
- OpenAI GPT-4o pricing: ~$2.50 / M input, ~$10.00 / M output
- Estimated: ~$1.00–1.60 for 50 questions × 3 passes (vs ~$0.09 for gptoss variant)
- The cost question is **whether the quality lift justifies 16× higher spend**

**Prerequisite:**
- `OPENAI_API_KEY` populated in `.env` at the repo root
- ≥$5 minimum credit on the OpenAI account (their fund-credit minimum)

## 1. Setup

In [1]:
import sys, os, json, time, textwrap, logging
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from dotenv import load_dotenv

os.environ["ANONYMIZED_TELEMETRY"] = "False"
logging.getLogger("chromadb.telemetry").setLevel(logging.ERROR)
logging.getLogger("chromadb").setLevel(logging.WARNING)

cwd = Path.cwd()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

from src.data.embedder import load_bge, best_device
from src.data.indices import load_chroma, load_bm25
from src.retrieval.hybrid import hybrid_top_k_with_text
from src.generation.openai_client import openai_complete_full
from src.generation.golden_prompts import (
    build_pass1_prompt, build_pass2_prompt, build_pass3_prompt,
    parse_json_with_reasoning_leak,
    validate_pass1, validate_pass2, validate_pass3,
)

# === A/B variant marker ===
CONSTRUCTOR_MODEL = "gpt-4o"      # OpenAI full GPT-4o (not mini)
PROVIDER_LABEL    = "openai"

CHUNKS_PATH       = REPO_ROOT / "data" / "processed" / "chunks.parquet"
MEDQA_4OPT_PATH   = REPO_ROOT / "data" / "processed" / "medqa_4opt.parquet"
CHROMA_DIR        = REPO_ROOT / "data" / "indices" / "chroma_textbooks"
BM25_PATH         = REPO_ROOT / "data" / "indices" / "bm25.pkl"
CACHE_DIR         = REPO_ROOT / "data" / "cache"
OUTPUT_DIR        = REPO_ROOT / "data" / "processed"

# A/B variant: gpt4o
PILOT_OUT     = OUTPUT_DIR / "golden_ragas_50_pilot_gpt4o.jsonl"
NEEDS_OUT     = OUTPUT_DIR / "golden_ragas_50_pilot_gpt4o_needs_review.jsonl"
DROPPED_OUT   = OUTPUT_DIR / "golden_ragas_50_pilot_gpt4o_dropped.jsonl"

# Constructor settings (per pass) — IDENTICAL to gptoss variant for fair A/B
PASS1_TEMP, PASS1_MAX_TOKENS = 0.0, 2048
PASS2_TEMP, PASS2_MAX_TOKENS = 0.2, 4096
PASS3_TEMP, PASS3_MAX_TOKENS = 0.0, 2048

# Hybrid retrieval k
RETRIEVAL_K = 10

# Sample size for the pilot — IDENTICAL seed/N for fair A/B
PILOT_N = 50
SEED = 42

print(f"Repo root:    {REPO_ROOT}")
print(f"Constructor:  {CONSTRUCTOR_MODEL}  ({PROVIDER_LABEL})")
print(f"Cache dir:    {CACHE_DIR}")
print(f"Outputs:      {PILOT_OUT.name}, {NEEDS_OUT.name}, {DROPPED_OUT.name}")


Repo root:    /Users/rajak/Workstation/Projects/myGitHub/thesis-project
Constructor:  gpt-4o  (openai)
Cache dir:    /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/cache
Outputs:      golden_ragas_50_pilot_gpt4o.jsonl, golden_ragas_50_pilot_gpt4o_needs_review.jsonl, golden_ragas_50_pilot_gpt4o_dropped.jsonl


## 2. Verify `OPENAI_API_KEY`

In [2]:
openai_key = os.environ.get("OPENAI_API_KEY")
assert openai_key, (
    "OPENAI_API_KEY missing — populate .env at the repo root and rerun. "
    "Get a key at https://platform.openai.com/api-keys (~$5 minimum credit)."
)
print(f"OPENAI_API_KEY: present (length {len(openai_key)})")

# Sanity: verify GPT-4o is reachable via models.list()
from openai import OpenAI
all_models = {m.id for m in OpenAI(api_key=openai_key).models.list().data}
assert CONSTRUCTOR_MODEL in all_models, (
    f"Constructor model {CONSTRUCTOR_MODEL!r} not reachable on this OpenAI account. "
    f"Visible: {sorted(m for m in all_models if 'gpt-4o' in m)}"
)
print(f"Constructor model {CONSTRUCTOR_MODEL!r}: reachable "
      f"({len(all_models)} models visible on this OpenAI account)")


OPENAI_API_KEY: present (length 164)
Constructor model 'gpt-4o': reachable (132 models visible on this OpenAI account)


## 3. Stratified sample 50 questions

Stratify by `meta_info` × question-length bucket (≤120 / 121–200 / >200 words). Force ~10 long-vignettes (>200 words) so the multi-hop architecture has a fair test surface. Random seed 42 for reproducibility.

In [3]:
medqa = pd.read_parquet(MEDQA_4OPT_PATH)
medqa["q_words"] = medqa["question"].str.split().str.len()

def length_bucket(n: int) -> str:
    if n <= 120: return "short"
    if n <= 200: return "medium"
    return "long"

medqa["length_bucket"] = medqa["q_words"].apply(length_bucket)

# Stratification: 6 cells = {step1, step2&3} × {short, medium, long}
# Force ~10 long total (~5 per meta_info), then split the remaining 40 across short+medium proportionally
N_LONG_PER_META = 5
N_REMAINDER = PILOT_N - 2 * N_LONG_PER_META   # 40

rng = np.random.RandomState(SEED)
parts = []
# Long vignettes
for meta in ["step1", "step2&3"]:
    pool = medqa[(medqa["meta_info"] == meta) & (medqa["length_bucket"] == "long")]
    parts.append(pool.sample(N_LONG_PER_META, random_state=rng))
# Remaining: stratified by meta × {short, medium} proportional to natural rates
remainder_pool = medqa[medqa["length_bucket"].isin(["short", "medium"])]
strata = remainder_pool.groupby(["meta_info", "length_bucket"])
total_remainder_pop = len(remainder_pool)
allocations = {}
for (meta, bucket), grp in strata:
    allocations[(meta, bucket)] = max(1, round(len(grp) / total_remainder_pop * N_REMAINDER))
# Adjust if rounding produced != 40
diff = N_REMAINDER - sum(allocations.values())
if diff != 0:
    biggest_key = max(allocations, key=lambda k: allocations[k])
    allocations[biggest_key] += diff
for (meta, bucket), n in allocations.items():
    pool = remainder_pool[(remainder_pool["meta_info"] == meta) &
                          (remainder_pool["length_bucket"] == bucket)]
    parts.append(pool.sample(n, random_state=rng))

dev_sample = pd.concat(parts).sample(frac=1, random_state=rng).reset_index(drop=True)
assert len(dev_sample) == PILOT_N, f"got {len(dev_sample)} rows"

print(f"Pilot sample: {len(dev_sample)} questions")
print("\nStrata breakdown:")
print(dev_sample.groupby(["meta_info", "length_bucket"]).size().unstack(fill_value=0))
print(f"\nLong-vignette count: {(dev_sample.length_bucket == 'long').sum()} / {PILOT_N}")


Pilot sample: 50 questions

Strata breakdown:
length_bucket  long  medium  short
meta_info                         
step1             5       5     17
step2&3           5      11      7

Long-vignette count: 10 / 50


## 4. Load shared infrastructure

In [4]:
chunks_df = pd.read_parquet(CHUNKS_PATH)
chunk_text_by_id = dict(zip(chunks_df.chunk_id, chunks_df.text))
chunk_book_by_id = dict(zip(chunks_df.chunk_id, chunks_df.book_name))

chroma_coll = load_chroma(CHROMA_DIR)
bm25_payload = load_bm25(BM25_PATH)
assert chroma_coll.count() == len(chunks_df)
assert len(bm25_payload['chunk_ids']) == len(chunks_df)

%time embedder = load_bge(device=best_device())

print(f"\nchunks_df rows : {len(chunks_df):,}")
print(f"ChromaDB count : {chroma_coll.count():,}")
print(f"BM25 chunk_ids : {len(bm25_payload['chunk_ids']):,}")
print(f"BGE device     : {embedder.device}")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


CPU times: user 220 ms, sys: 573 ms, total: 792 ms
Wall time: 8.32 s

chunks_df rows : 67,599
ChromaDB count : 67,599
BM25 chunk_ids : 67,599
BGE device     : mps:0


## 5. Define the 3-pass constructor pipeline

Pure orchestration — wraps Stages B → E into one function. Every call goes through `openai_complete_full` so all 3 passes are disk-cached and re-runs are free.

In [5]:
def construct_one_row(row: pd.Series) -> dict:
    """Run Hybrid retrieval + 3 constructor passes for a single question.

    Returns an audit dict with all intermediate state — cell §7 will collect these
    into a list and §8 (audit) + §9 (save) consume them.
    """
    options = json.loads(row["options_json"])
    correct_letter = row["answer_idx"]
    correct_text = options[correct_letter]
    question_text = row["question"]

    # === Stage B: Hybrid retrieval (construction-time bias: prepend the answer to the query) ===
    construction_query = f"{question_text}  Answer: {correct_text}"
    candidates = hybrid_top_k_with_text(
        construction_query,
        embedder_model=embedder,
        chroma_coll=chroma_coll,
        bm25_payload=bm25_payload,
        chunks_df=chunks_df,
        k=RETRIEVAL_K,
    )

    audit = {
        "question_id":   int(row.name),
        "meta_info":     row["meta_info"],
        "length_bucket": row["length_bucket"],
        "question":      question_text,
        "options":       options,
        "answer_idx":    correct_letter,
        "candidate_ids": [c["chunk_id"] for c in candidates],
        "stage_b_n_candidates": len(candidates),
    }

    # === Stage C: Pass 1 — evidence selection ===
    p1_prompt = build_pass1_prompt(question_text, options, correct_letter, correct_text, candidates)
    p1_payload, p1_cached = openai_complete_full(
        p1_prompt, model=CONSTRUCTOR_MODEL,
        temperature=PASS1_TEMP, max_tokens=PASS1_MAX_TOKENS,
        cache_dir=CACHE_DIR,
    )
    audit["pass1_payload"] = p1_payload
    audit["pass1_cached"] = p1_cached
    p1 = parse_json_with_reasoning_leak(p1_payload["text"])
    audit["pass1_parsed"] = p1
    if p1 is None:
        audit["error"] = "pass1_json_malformed"; return audit
    ok, reason = validate_pass1(p1)
    if not ok:
        audit["error"] = f"pass1_schema:{reason}"; return audit

    # Resolve gold context from selected_chunk_ids
    selected_ids = p1["selected_chunk_ids"]
    gold_context = []
    for cid in selected_ids:
        if cid in chunk_text_by_id:
            gold_context.append({
                "chunk_id":  cid,
                "text":      chunk_text_by_id[cid],
                "book_name": chunk_book_by_id.get(cid, ""),
            })
    audit["gold_context_resolved"] = len(gold_context)
    if len(gold_context) == 0:
        audit["error"] = "pass1_no_resolvable_chunk_ids"; return audit
    audit["evidence_keywords"] = p1.get("evidence_keywords", [])
    audit["is_evidence_sufficient"] = bool(p1.get("is_evidence_sufficient"))

    # === Stage D: Pass 2 — reference answer + explanation ===
    p2_prompt = build_pass2_prompt(question_text, options, correct_letter, correct_text, gold_context)
    p2_payload, p2_cached = openai_complete_full(
        p2_prompt, model=CONSTRUCTOR_MODEL,
        temperature=PASS2_TEMP, max_tokens=PASS2_MAX_TOKENS,
        cache_dir=CACHE_DIR,
    )
    audit["pass2_payload"] = p2_payload
    audit["pass2_cached"] = p2_cached
    p2 = parse_json_with_reasoning_leak(p2_payload["text"])
    audit["pass2_parsed"] = p2
    if p2 is None:
        audit["error"] = "pass2_json_malformed"; return audit
    ok, reason = validate_pass2(p2)
    if not ok:
        audit["error"] = f"pass2_schema:{reason}"; return audit

    # === Stage E: Pass 3 — validation ===
    p3_prompt = build_pass3_prompt(
        question_text, correct_letter, correct_text, gold_context,
        p2["reference_answer"], p2["reference_explanation"],
    )
    p3_payload, p3_cached = openai_complete_full(
        p3_prompt, model=CONSTRUCTOR_MODEL,
        temperature=PASS3_TEMP, max_tokens=PASS3_MAX_TOKENS,
        cache_dir=CACHE_DIR,
    )
    audit["pass3_payload"] = p3_payload
    audit["pass3_cached"] = p3_cached
    p3 = parse_json_with_reasoning_leak(p3_payload["text"])
    audit["pass3_parsed"] = p3
    if p3 is None:
        audit["error"] = "pass3_json_malformed"; return audit
    ok, reason = validate_pass3(p3)
    if not ok:
        audit["error"] = f"pass3_schema:{reason}"; return audit

    # Successful audit row — assemble the golden record candidate
    audit["golden_row"] = {
        "question_id":                          int(row.name),
        "meta_info":                            row["meta_info"],
        "question":                             question_text,
        "options":                              options,
        "gold_answer_letter":                   correct_letter,
        "gold_answer_text":                     correct_text,
        "gold_chunks":                          [c["chunk_id"] for c in gold_context],
        "gold_context":                         [c["text"] for c in gold_context],
        "evidence_keywords":                    p1.get("evidence_keywords", []),
        "reference_answer":                     p2["reference_answer"],
        "reference_explanation":                p2["reference_explanation"],
        "why_other_options_are_less_suitable": p2["why_other_options_are_less_suitable"],
        "hallucination_check_points":           p2["hallucination_check_points"],
        "question_type":                        p2["question_type"].lower(),
        "requires_multihop":                    p2["requires_multihop"].lower(),
        "evidence_relevance_score":             p3["evidence_relevance_score"],
        "faithfulness_score":                   p3["faithfulness_score"],
        "explanation_quality_score":            p3["explanation_quality_score"],
        "hallucination_risk":                   p3["hallucination_risk"],
        "final_status":                         p3["final_status"],
    }
    return audit


## 6. Tiny smoke — 1 question through all 3 passes

Verify each pass returns parseable, schema-valid JSON before launching the 50-row loop. **First call costs ~$0.02; cached re-runs are $0.**

In [6]:
smoke = construct_one_row(dev_sample.iloc[0])

print(f"Pass 1 cached={smoke.get('pass1_cached')}, gold_context resolved={smoke.get('gold_context_resolved')}/3")
print(f"Pass 2 cached={smoke.get('pass2_cached')}")
print(f"Pass 3 cached={smoke.get('pass3_cached')}")
print(f"\nKeys in audit: {list(smoke.keys())}")

if "error" in smoke:
    print(f"\n✗ ERROR: {smoke['error']}")
    print(f"--- Pass 1 raw text (first 500) ---")
    print(smoke.get("pass1_payload", {}).get("text", "")[:500])
else:
    print("\n✓ smoke passed — all 3 passes produced valid JSON")
    g = smoke["golden_row"]
    print(f"\n--- sample reference answer ---")
    print(f"Q: {g['question'][:200]}...")
    print(f"A ({g['gold_answer_letter']}): {g['gold_answer_text']}")
    print(f"reference_answer:      {g['reference_answer']}")
    print(f"reference_explanation: {g['reference_explanation'][:300]}...")
    print(f"hallucination_check_points: {g['hallucination_check_points']}")
    print(f"question_type:         {g['question_type']}")
    print(f"requires_multihop:     {g['requires_multihop']}")
    print(f"final_status:          {g['final_status']}  (relevance={g['evidence_relevance_score']}, faithfulness={g['faithfulness_score']}, quality={g['explanation_quality_score']})")


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Pass 1 cached=False, gold_context resolved=2/3
Pass 2 cached=False
Pass 3 cached=False

Keys in audit: ['question_id', 'meta_info', 'length_bucket', 'question', 'options', 'answer_idx', 'candidate_ids', 'stage_b_n_candidates', 'pass1_payload', 'pass1_cached', 'pass1_parsed', 'gold_context_resolved', 'evidence_keywords', 'is_evidence_sufficient', 'pass2_payload', 'pass2_cached', 'pass2_parsed', 'pass3_payload', 'pass3_cached', 'pass3_parsed', 'golden_row']

✓ smoke passed — all 3 passes produced valid JSON

--- sample reference answer ---
Q: A group of neurologists develop a new blood test for Alzheimer's. They are optimistic about the test, as they have found that for any given patient, the test repeatedly produces very similar results. ...
A (B): Reliable
reference_answer:      The new test would most accurately be described as reliable because it produces very similar results for any given patient.
reference_explanation: Reliability refers to the consistency of a test in producing si

## 7. Run all 50

Sequential loop with progress prints every 5 questions. **Cost: ~$1.00–1.60 total** (single payment, then cache makes restarts free). OpenAI's tier-1 rate limits are higher than Groq's free tier so wall time should be ~3–6 min.

In [7]:
results: list[dict] = []
t0_total = time.time()
for i, (_, row) in enumerate(dev_sample.iterrows()):
    t0 = time.time()
    audit = construct_one_row(row)
    elapsed = time.time() - t0
    results.append(audit)
    if (i + 1) % 5 == 0 or i == 0:
        cached_count = sum(1 for r in results
                            if r.get("pass1_cached") and r.get("pass2_cached") and r.get("pass3_cached"))
        err_count = sum(1 for r in results if "error" in r)
        print(f"[{i+1:>3}/{PILOT_N}]  {elapsed:5.1f}s  cached_full={cached_count}  errors={err_count}")
elapsed_total = time.time() - t0_total
print(f"\nDone. Wall time: {elapsed_total/60:.1f} min for {PILOT_N} questions.")


[  1/50]    2.6s  cached_full=1  errors=0
[  5/50]   12.4s  cached_full=1  errors=0
[ 10/50]   12.9s  cached_full=1  errors=0
[ 15/50]   12.4s  cached_full=1  errors=0
[ 20/50]    8.9s  cached_full=1  errors=0
[ 25/50]   13.7s  cached_full=1  errors=0
[ 30/50]   21.5s  cached_full=1  errors=0
[ 35/50]   12.6s  cached_full=1  errors=0
[ 40/50]   16.4s  cached_full=1  errors=0
[ 45/50]    9.0s  cached_full=1  errors=0
[ 50/50]   10.5s  cached_full=1  errors=0

Done. Wall time: 11.0 min for 50 questions.


## 8. Stage F — Automated audit

Pure-Python checks (no LLM):
1. Every `chunk_id` cited in `gold_chunks` resolves in `chunks.parquet`
2. The gold answer text appears (case-insensitive) in `reference_answer`
3. At least one `evidence_keyword` appears in the gold context

Failures here downgrade `final_status` from `accepted` → `needs_review`.

In [8]:
def audit_row(audit: dict) -> tuple[str, list[str]]:
    """Return (status, audit_flags). status is the final status after audit."""
    if "error" in audit:
        return "rejected", [audit["error"]]

    g = audit["golden_row"]
    flags = []

    # 1. Every chunk_id must resolve
    unresolved = [cid for cid in g["gold_chunks"] if cid not in chunk_text_by_id]
    if unresolved:
        flags.append(f"unresolved_chunk_ids:{len(unresolved)}")

    # 2. Gold answer text appears in reference_answer
    if g["gold_answer_text"].lower() not in g["reference_answer"].lower():
        flags.append("gold_answer_not_in_reference")

    # 3. At least one keyword appears in gold context
    context_lower = " ".join(g["gold_context"]).lower()
    keyword_hits = sum(1 for kw in g["evidence_keywords"] if kw.lower() in context_lower)
    if keyword_hits == 0:
        flags.append("no_keyword_in_context")
    audit["audit_keyword_hits"] = keyword_hits

    # Apply downgrade
    pass3_status = g["final_status"]
    if flags:
        if pass3_status == "accepted":
            return "needs_review", flags
        return pass3_status, flags  # keep needs_review/rejected as-is
    return pass3_status, flags

audit_outcomes = []
for r in results:
    status, flags = audit_row(r)
    r["post_audit_status"] = status
    r["audit_flags"] = flags
    audit_outcomes.append(status)

print("Post-audit status distribution:")
print(pd.Series(audit_outcomes).value_counts().to_string())


Post-audit status distribution:
accepted        20
needs_review    19
rejected        11


## 9. Stage G — Save accepted / needs_review / dropped

In [9]:
def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

accepted_rows  = [r["golden_row"] | {"audit_flags": []} for r in results
                  if r.get("post_audit_status") == "accepted"]
review_rows    = [r["golden_row"] | {"audit_flags": r.get("audit_flags", [])} for r in results
                  if r.get("post_audit_status") == "needs_review"]
dropped_rows   = []
for r in results:
    if r.get("post_audit_status") == "rejected":
        rec = r.get("golden_row", {})
        rec["error"]       = r.get("error", "")
        rec["audit_flags"] = r.get("audit_flags", [])
        rec["question_id"] = r.get("question_id")
        dropped_rows.append(rec)

write_jsonl(PILOT_OUT,   accepted_rows)
write_jsonl(NEEDS_OUT,   review_rows)
write_jsonl(DROPPED_OUT, dropped_rows)

for path, rows in [(PILOT_OUT, accepted_rows), (NEEDS_OUT, review_rows), (DROPPED_OUT, dropped_rows)]:
    print(f"  {path.name:<50}  {len(rows)} rows  ({path.stat().st_size/1024:.1f} KB)")


  golden_ragas_50_pilot_gpt4o.jsonl                   20 rows  (137.0 KB)
  golden_ragas_50_pilot_gpt4o_needs_review.jsonl      19 rows  (116.4 KB)
  golden_ragas_50_pilot_gpt4o_dropped.jsonl           11 rows  (62.5 KB)


## 10. Quality gates + real-cost summary

This is the final answer to the cost-vs-quality question. Numbers here will be propagated back into `plan.md`, `docs/tech_stack.md`, and the project memory.

In [10]:
# === Quality gates ===
n          = len(results)
n_pass1_ok = sum(1 for r in results if r.get("pass1_parsed") is not None)
n_pass2_ok = sum(1 for r in results if r.get("pass2_parsed") is not None)
n_pass3_ok = sum(1 for r in results if r.get("pass3_parsed") is not None)
n_accepted = sum(1 for r in results if r.get("post_audit_status") == "accepted")
n_review   = sum(1 for r in results if r.get("post_audit_status") == "needs_review")
n_rejected = sum(1 for r in results if r.get("post_audit_status") == "rejected")

# JSON malformation rate
malformed = []
for r in results:
    for k in ("pass1_parsed", "pass2_parsed", "pass3_parsed"):
        if k in r and r[k] is None:
            malformed.append(k)
malformation_rate = len(malformed) / (n * 3)

# Pass-1 sufficiency
n_with_p1 = sum(1 for r in results if r.get("pass1_parsed") is not None)
n_sufficient = sum(1 for r in results if r.get("is_evidence_sufficient") is True)
sufficiency_rate = (n_sufficient / n_with_p1) if n_with_p1 else 0.0

# Multi-hop rate (FIXED: divide by total n, not n_accepted)
multihop_yes = sum(1 for r in results if r.get("golden_row", {}).get("requires_multihop") == "yes")
multihop_rate = multihop_yes / n

# All chunk_ids resolve check
all_resolve = all(
    all(cid in chunk_text_by_id for cid in r["golden_row"]["gold_chunks"])
    for r in results if "golden_row" in r
)

print(f"=== QUALITY GATES — {CONSTRUCTOR_MODEL} ===")
gates = [
    ("Accept rate           >= 80%",  f"{n_accepted}/{n} = {n_accepted/n:.1%}",  n_accepted/n >= 0.80),
    ("JSON malformation     <  5%",  f"{len(malformed)}/{n*3} = {malformation_rate:.2%}",  malformation_rate < 0.05),
    ("Pass-1 sufficiency    >= 90%", f"{n_sufficient}/{n_with_p1} = {sufficiency_rate:.1%}", sufficiency_rate >= 0.90),
    ("All chunk_ids resolve",         "yes" if all_resolve else "no",                       all_resolve),
    ("requires_multihop yes <  60%", f"{multihop_yes}/{n} = {multihop_rate:.1%}",            multihop_rate < 0.60),
]
for name, value, ok in gates:
    print(f"  {'✓' if ok else '✗'}  {name:<35}  {value}")
print()

# === Real cost ===
total_in  = 0
total_out = 0
total_cached = 0
for r in results:
    for pname in ("pass1_payload", "pass2_payload", "pass3_payload"):
        p = r.get(pname)
        if p is None: continue
        u = p.get("usage", {}) or {}
        total_in  += int(u.get("prompt_tokens") or 0)
        total_out += int(u.get("completion_tokens") or 0)
    for cname in ("pass1_cached", "pass2_cached", "pass3_cached"):
        if r.get(cname): total_cached += 1

# OpenAI GPT-4o pricing: $2.50/M in, $10.00/M out
openai_in_rate  = 2.50  / 1_000_000
openai_out_rate = 10.00 / 1_000_000
cost_paid_openai = total_in * openai_in_rate + total_out * openai_out_rate

# Groq gpt-oss-120b pricing for comparison
cost_groq_paid = total_in * (0.15 / 1_000_000) + total_out * (0.60 / 1_000_000)

print("=== REAL COST ===")
print(f"  total prompt tokens   : {total_in:,}")
print(f"  total completion tokens: {total_out:,}")
print(f"  total tokens          : {total_in + total_out:,}")
print(f"  cache hits (per call) : {total_cached} / {n*3}")
print()
print(f"  Cost on OpenAI (this run): ${cost_paid_openai:.4f}  (${cost_paid_openai*6:.2f} extrapolated to 300 rows)")
print(f"  Same workload on Groq gpt-oss-120b paid: ${cost_groq_paid:.4f}  (${cost_groq_paid*6:.2f} for 300)")
print(f"  Ratio: GPT-4o is {cost_paid_openai/cost_groq_paid:.1f}x more expensive than gpt-oss-120b on Groq paid tier")


=== QUALITY GATES — gpt-4o ===
  ✗  Accept rate           >= 80%         20/50 = 40.0%
  ✓  JSON malformation     <  5%          0/150 = 0.00%
  ✓  Pass-1 sufficiency    >= 90%         49/50 = 98.0%
  ✓  All chunk_ids resolve                yes
  ✗  requires_multihop yes <  60%         33/50 = 66.0%

=== REAL COST ===
  total prompt tokens   : 336,513
  total completion tokens: 27,201
  total tokens          : 363,714
  cache hits (per call) : 3 / 150

  Cost on OpenAI (this run): $1.1133  ($6.68 extrapolated to 300 rows)
  Same workload on Groq gpt-oss-120b paid: $0.0668  ($0.40 for 300)
  Ratio: GPT-4o is 16.7x more expensive than gpt-oss-120b on Groq paid tier


## 11. Sample outputs (eyeball check)

Print 2 accepted + 1 needs_review (if any) so you can sanity-check reference answer quality before scaling to the 250-row production build.

In [11]:
def show(r):
    g = r["golden_row"]
    print(f"--- {r.get('post_audit_status')}  audit_flags={r.get('audit_flags')}  "
          f"P3 scores: rel={g['evidence_relevance_score']} faith={g['faithfulness_score']} qual={g['explanation_quality_score']} ---")
    print(f"Q ({g['meta_info']}): {g['question'][:180]}...")
    print(f"A ({g['gold_answer_letter']}): {g['gold_answer_text']}")
    print(f"reference_answer:    {g['reference_answer']}")
    print(f"reference_explan:    {g['reference_explanation'][:280]}...")
    print(f"check points:        {g['hallucination_check_points']}")
    print(f"requires_multihop:   {g['requires_multihop']}    question_type: {g['question_type']}")
    print()

accepted = [r for r in results if r.get("post_audit_status") == "accepted"]
review   = [r for r in results if r.get("post_audit_status") == "needs_review"]

for r in accepted[:2]:
    show(r)
if review:
    print("=== needs_review sample ===")
    show(review[0])


--- accepted  audit_flags=[]  P3 scores: rel=4 faith=5 qual=5 ---
Q (step1): A group of neurologists develop a new blood test for Alzheimer's. They are optimistic about the test, as they have found that for any given patient, the test repeatedly produces ve...
A (B): Reliable
reference_answer:    The new test would most accurately be described as reliable because it produces very similar results for any given patient.
reference_explan:    Reliability refers to the consistency of a test in producing similar results under consistent conditions. The gold context mentions that the neurologists found the test repeatedly produces very similar results, indicating reliability. However, the test results are not consistent ...
check points:        ['The test produces very similar results for any given patient.', 'The test results are not consistent with the gold standard of diagnosis.', 'Reliability refers to the consistency of a test in producing similar results.', 'Validity is determined by co

---

**Done.** Send the §10 output to me — I'll merge it with the gptoss results into a side-by-side comparison table, then propagate the winning constructor + real numbers into `plan.md` / `docs/tech_stack.md` / `docs/architecture.md` / memory and write `docs/output_notes/04_notebook_output.md`.

**Compare against the gptoss variant:**
- Run [`04_golden_dataset_gptoss.ipynb`](04_golden_dataset_gptoss.ipynb) (already cached — instant re-run)
- Run this one (~$1–1.60 first time)
- Decide based on `accept rate (gpt4o) - accept rate (gptoss)` against `cost difference`. If GPT-4o gives +30% accept rate for $1.50, that's worth it. If only +10%, gpt-oss is the call (after prompt-tuning).